# NLP Feature Extraction
- Part-of-Speech Tagging
- Term Frequency-Inverse Document Frequency
- Sentiment Analysis
- Pretrained Word Embeddings
- Custom Word Embbeding Model
- Speaker / Speaker history

## Import Dependencies

In [1]:
import spacy
import pandas as pd
import numpy as np
import os
from sklearn.feature_extraction.text import TfidfVectorizer
import re
from typing import List
from utils import load_model_and_tokenizer, load_inference_data
from sentiment_analysis_eval import get_predictions

# Helper Functions


In [2]:
def load_df(filepath: str) -> pd.DataFrame:
    """Function to load a pd.DataFrame from a given filepath"""
    try:
        if filepath.endswith('.csv'):
            df = pd.read_csv(filepath)
        elif filepath.endswith('.xlsx'):
            df = pd.read_excel(filepath)
        else:
            raise ValueError("Unsupported file format. Please use .csv or .xlsx")
    except FileNotFoundError:
        raise FileNotFoundError(f"File not found: {filepath}")
    return df

def load_sentences(filepath: str, text_column: str = "sentence") -> pd.Series:
    """Function to load a column from a pd.DataFrame into a list
    Author: Nick Belterman"""
    df = load_df(filepath)
    return df[text_column]


## Create Variables

In [ ]:
language_code_mapping = {
    'nl': 'Dutch',
    'en': 'English',
    'de': 'German',
    'fr': 'French',
    'es': 'Spanish',
    'it': 'Italian',
    'pt': 'Portuguese',
    'ru': 'Russian',
    'zh': 'Chinese',
    'ja': 'Japanese',
    'ko': 'Korean',
    'ar': 'Arabic',
    'hi': 'Hindi',
    'bn': 'Bengali',
    'ur': 'Urdu',
    'vi': 'Vietnamese',
    'tr': 'Turkish',
    'sv': 'Swedish',
    'da': 'Danish',
    'no': 'Norwegian',
    'fi': 'Finnish',
    'pl': 'Polish',
    'cs': 'Czech',
    'hu': 'Hungarian',
    'ro': 'Romanian',
    'el': 'Greek',
    'he': 'Hebrew',
}

language = 'nl'  # Change this to the desired language code
transcription_file = "..\\data\\transcription\\csv\\transcription.csv"  # Change this to your transcription file path
output_file = '..\\data\\features\\NLP_features_output.csv'  # Change this to your desired output file path
print(f"Download model using `poetry run python -m spacy download {language}_core_news_lg` if not already installed.")

Download model using `poetry run python -m spacy download nl_core_news_lg` if not already installed.


## Function to Load Spacy Model

In [4]:

def load_spacy_model(
    language: str = 'nl', 
    text_column: str = 'Sentence',
    ) -> spacy.language.Language:
    """Load the spaCy model for the specified language.
    Args:
        language (str): Language code (default is 'nl' for Dutch).
    Returns:
        spacy.language.Language: Loaded spaCy language model.
    Raises:
        ValueError: If the language code is not supported.
    Authors: Floris Lokhorst, Nick Belterman
    """
    model_name = f"{language}_core_news_lg"
    try:
        language_name = language_code_mapping.get(language)
    except KeyError:
        raise ValueError(f"Unsupported language code: {language}")

    # Configuration
    CONFIG = {
        'spacy_model': model_name,
        'language': language_name,
        'text_column': text_column,  
    }

    return spacy.load(CONFIG['spacy_model']), CONFIG

## Functions to Extract POS tags

In [5]:
def extract_save_pos_tags(
    input_filepath: str, 
    nlp_model: spacy.language.Language, \
    output_filepath: str = '..\\data\\features\\NLP_features_output.csv',
    config: dict = None
) -> None:
    """Extract POS tags from text.
    
    Authors: Floris Lokhorst, Nick Belterman
    """
    df  = load_df(input_filepath)
    text_column = config.get('text_column', 'Sentence') if config else 'Sentence'
    if not os.path.exists(os.path.dirname(output_filepath)):
        os.makedirs(os.path.dirname(output_filepath))
    data = load_sentences(input_filepath, text_column=text_column)
    # Process each sentence and store the POS tags
    df['POS_tags'] = data.astype(str).apply(
        lambda text: ' '.join(f"{token.text}_{token.pos_}" for token in nlp_model(text.strip())) if text and text.strip() else ''
    )
    
    # Placeholders for start_time and end_time
    data['start_time'] = np.nan
    data['end_time'] = np.nan

    # Save the DataFrame to a CSV file
    try:
        df.to_csv(output_filepath, index=False)
        print(f"Successfully processed {len(data)} sentences and saved the output to {output_filepath}")
    except Exception as e:
        print(f"Error saving the CSV file: {e}")

# Code to extract and save POS tags using spacy

In [6]:
nlp, config = load_spacy_model(text_column='sentence')
extract_save_pos_tags(transcription_file, nlp, output_file, config)


Successfully processed 1085 sentences and saved the output to ..\data\features\NLP_features_output.csv


# TF-IDF

# Function to load and fit TF-IDF Vectorizer on our chosen episode 

In [7]:
def load_fit_tfidf(
    sentences: List[str], 
) -> (np.ndarray, List[str]):
    """Load and fit a TF-IDF vectorizer on the provided sentences.
    
    Args:
        sentences (List[str]): List of sentences to fit the TF-IDF vectorizer.
    
    Authors: Jeroen Brower, Nick Belterman
    """
    vectorizer = TfidfVectorizer(
        max_features=2000,         
        min_df=1,                    
        max_df=1.0,                  
        lowercase=True,
        stop_words=None,             
        token_pattern=r'\b[a-záéíóúàèìòùâêîôûäëïöüç]+\b',  
    )
    print("Fitting TF-IDF...")
    tfidf_matrix = vectorizer.fit_transform(sentences)
    vocab = vectorizer.get_feature_names_out()
    print(f"Total vocabulary: {len(vocab)}")
    return tfidf_matrix, vocab

# Function to add the TF-IDF matrix to our feature extraction output file

In [8]:
def create_tfidf_dataframe(
    tfidf_matrix: np.ndarray, 
    vocab: List[str],
    output_filepath: str = '..\\data\\features\\NLP_features_output.csv',
    text_column: str = 'sentence',
    readable: bool = False,
) -> pd.DataFrame:
    """Create a DataFrame from the TF-IDF matrix and vocabulary.
    
    Args:
        tfidf_matrix (np.ndarray): The TF-IDF matrix.
        vocab (List[str]): The vocabulary list.
    
    Returns:
        pd.DataFrame: DataFrame with TF-IDF features.
    
    Authors: Jeroen Brower, Nick Belterman
    """
    print(tfidf_matrix[0])
    if output_filepath.endswith('.csv'):
        df = pd.read_csv(output_filepath)
    elif output_filepath.endswith('.xlsx'):
        df = pd.read_excel(output_filepath)

    if readable:
        print("Creating readable TF-IDF format...")
        # Create readable format
        tfidf_readable = []
        for i in range(len(sentences)):
            sentence = sentences[i]
            vector = tfidf_matrix[i].toarray()[0]
            
            # Get ALL words with their scores
            all_scores = [(vocab[j], score) for j, score in enumerate(vector) if score > 0]
            all_scores.sort(key=lambda x: x[1], reverse=True)
            
            # Format top words
            if all_scores:
                formatted = [f"{word}({score:.3f})" for word, score in all_scores[:8]]
                result = "; ".join(formatted)
            else:
                result = "EMPTY"
            
            tfidf_readable.append(result)
        df['TF_IDF'] = tfidf_readable
    else:
        print("Creating non-readable TF-IDF format...")
        # Non-readable format (sparse)
        tfidf_list = []
        for i in range(len(sentences)):
            vector = tfidf_matrix[i].toarray()[0]
            indices = [j for j, score in enumerate(vector) if score > 0]
            scores = [vector[j] for j in indices]
            pairs = [f"{vocab[j]}({score:.3f})" for j, score in zip(indices, scores)]
            result = "; ".join(pairs) if pairs else "EMPTY"
            tfidf_list.append(result)
        df['TF_IDF'] = tfidf_list


    # Dropna
    df.dropna(subset=[text_column], inplace=True)
    # Add to dataframe
    
    df.to_csv(output_filepath, index=False)
    print(f"Saved TF-IDF features to {output_filepath}")

# Code to extract and save TF-IDF terms using Scikit-learn

In [9]:
sentences = load_sentences(output_file, text_column='sentence').dropna()
tfidf_matrix, vocab = load_fit_tfidf(sentences)
create_tfidf_dataframe(tfidf_matrix, vocab, output_file)

Fitting TF-IDF...
Total vocabulary: 1385
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6 stored elements and shape (1, 1385)>
  Coords	Values
  (0, 539)	0.19359225234944083
  (0, 1311)	0.4212732524598009
  (0, 291)	0.33455176423019356
  (0, 748)	0.36345892697339593
  (0, 889)	0.5201074946806784
  (0, 868)	0.5201074946806784
Creating non-readable TF-IDF format...
Saved TF-IDF features to ..\data\features\NLP_features_output.csv


In [10]:
df = pd.read_csv(output_file)
print(df['TF_IDF'][0])

eigenlijk(0.335); ik(0.194); mijn(0.363); overdraaien(0.520); penningmeesterschap(0.520); wil(0.421)


# Sentiment Analysis

Model can be found in [Models_nlp_2526](https://edubuas-my.sharepoint.com/:f:/g/personal/225538_buas_nl/EuvOvnKbcLdPnsq8LGqDbJ0BP0DLEER7PMAwyS1bOy4JZg?e=olbDPM) and should be saved in `nlp_cia\\models\\` 

In [11]:
df = df.dropna(subset=['sentence'])
# subset = df.iloc[:15]
tokenizer, model = load_model_and_tokenizer(model_dir=r"..\\models\\Model_14e03c00")
sentences, df = load_inference_data(tokenizer, inference_data=df)
preds = get_predictions(model, sentences)
print(f"Predictions: {preds}")
df.to_csv(output_file, index=False)

Loading model and tokenizer from: ..\\models\\Model_14e03c00
c:\Users\nickb\OneDrive\Documenten\GitHub\fae2-nlpr-group-group-4-1\nlp_cia\src
texts = ['Ik wil eigenlijk mijn penningmeesterschap overdraaien', 'Wie stemt er op', 'Cecile, Malfik en Klaas', 'En toen kreeg ik twee stemmen', 'Terwijl ik totaal niet met geld om kan gaan', 'Het gevaar schuilt hier dicht onder het wateroppervlak', 'Drie jokers en min 500 euro', 'Ja zeg, dit is de hoofdvlees', 'Daarboven staat een kist', 'Dus ik weet zeker dat ik ze alle vier gehad heb', 'Maar twee schoten, h', 'O, kwaad', 'Wat geld', 'Wat moet een mens zonder Ellie Lust', 'Ellie', 'Kwaad', 'Het lijkt wel een uitgebreid gezin, de kandidatengroep van dit jaar', 'Lekker huiselijk rollen ze door het spel heen', 'Schone schijn misschien, want de openheid is ingeruild voor mysteries', 'Er borrelt van alles onder het oppervlak dat het verse gezinsgeluk kan verstoren', 'Bij iedere afvaller die vertrekt, valt het afscheid de groep zwaarder', 'Al leek Mar